# మానవ-ఇన్-ది-లూప్: ప్రీ-ఆక్షన్ గేట్లు, రిస్క్ టియరింగ్, మరియు ఆడిట్ లాగింగ్

ఈ పాఠం కోసం README మానవ-ఇన్-ది-లూప్ ని పరిచయం చేస్తుంది ఒక చిన్న స్నిపెట్‌తో, అది యూజర్‌ని అడుగుతుంది `APPROVE` లేదా `REJECT` అనే నిర్ణయం తీసుకోవాలని, ఏజెంట్ ఇప్పటికే స్పందన ఇవ్వగానే. ఆ నమూనా మంచి ప్రారంభ బిందువు, కానీ ప్రొడక్షన్ HITL అమలులో మూడు అదనపు భాగాలు సాధారణంగా అవసరం అవుతాయి:

1. ఏజెంట్ ఒక ప్రమాదకరమైన దశను ప్రదర్శించక ముందు నడిచే **ప్రీ-ఆక్షన్ గేట్**, తద్వారా ఖర్చు, తిరగకపోటు, మరియు ఆలస్యం నియంత్రణలో ఉంటాయి.
2. **రిస్క్ టియరింగ్**, అంటే తక్కువ-ప్రమాద చర్యలు స్వయంచాలకంగా అమలవుతాయి, మిడియం-ప్రమాద చర్యలు బ్యాచ్ ఆమోదించబడతాయి, ఇంకా మాత్రమే హై-రిస్క్ చర్యలు మానవుడి ఆప్తవుతాయి.
3. **ఆడిట్ లాగ్ మరియు రివిజన్ లూప్**, అందులో ప్రతి గేట్ నిర్ణయం JSONL గా రికార్డు చేయబడుతుంది, మరియు తిరస్కరణ ఒక క్రమం నిర్ణయంతో ఏజెంట్‌ను మళ్ళీ ప్రాంప్ట్ చేస్తుంది కేవలం `Revising...` ముద్రించకుండా.

ఈ నోట్బుక్ ఈ మూడు అంశాలను ఒకే ప్రాథమిక మూలాలతో నిర్మిస్తుంది `06-system-message-framework.ipynb` లాగా. ఇది ఎండ్టు-ఎండ్కి `DEMO_MODE = True` లో నడుస్తుంది (ఎలాంటి ఇంటరాక్టివ్ ఇన్‌పుట్ అవసరం లేకుండా) లేదా నిజమైన `input()` ప్రాంప్ట్‌లతో, `DEMO_MODE = False` ఉన్నప్పుడు. గమనిక: DEMO_MODE లో మూడవ లక్ష్యం రీట్రై స్క్రిప్టెడ్ విధంగా ఉంటుంది కాబట్టి లూప్ యంత్రాంగం పూర్తిగా కనబడుతుంది. వాస్తవ రివిజన్ ఆధారిత పునర్మూల్యాంకనం కోసం `DEMO_MODE = False` మరియు ఒక ఆపరేటర్ అవసరం.

**వెడల్పు వెలుపల (ఇతర పాఠాలలో నిర్వహించబడుతుంది):** ఆథెంటికేషన్ మరియు ప్రవేశ నియంత్రణ (Lesson 06 README threat 2), టూల్-కాల్ మిడ్‌ల్వేర్ (Lesson 14 MAF లో లోతైన విశ్లేషణ), బహుళ ఏజెంట్ వాదన నమూనాలు.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## నమూనా 1: ముందు-క్రియ గేట్

README యొక్క HITL స్నిపెట్ మొదట ఏజెంట్‌ను పిలుస్తుంది, తరువాత అవుట్పుట్‌ను ఆమోదించమని యూజర్‌ను కోరుతుంది. అది ఒక **పోస్ట్-క్రియ** ఫ్లో. ఏజెంట్ ఇప్పటికే అమలుచేసింది, కాబట్టి LLM కాల్ ఛార్జీ ఇప్పటికే చెల్లించబడింది, మరియు ఏ సైడ్ ఎఫెక్ట్ (సెండ్ చేయబడిన ఇమెయిల్, రాసిన డేటాబేస్ రో, పోస్ట్ చేసిన కామెంట్) ఇప్పటికే జరిగింది.

ఒక **ముందు-క్రియ** ఫ్లో ప్రమాదకరమైన దశ అమలుచేసే ముందు గేట్‌ను చేర్చుతుంది. ఏజెంట్ చర్యను ప్రతిపాదిస్తుంది, గేట్ అమలు చేయాలా నిర్ణయించుకుంటుంది, మరియు ఆమోదం వచ్చినప్పుడు మాత్రమే సైడ్ ఎఫెక్ట్ జరుగుతుంది.

| అంశం | పోస్ట్-క్రియ ఆమోదం (README స్నిపెట్) | ముందు-క్రియ గేట్ (ఈ నోట్బుక్) |
|---|---|---|
| ఆమోదం ఎప్పుడు ఉంటుంది? | ఏజెంట్ అవుట్పుట్ ఉత్పత్తి చేసిన తర్వాత | ఏ సైడ్-ఎఫెక్ట్ అమలు కాబోతున్న దాని ముందు |
| తిరస్కరణపై LLM ఖర్చు | ఇప్పటికే చెల్లించబడింది | ప్రతిపాదన కోసం మాత్రమే చెల్లించబడుతుంది, చర్యకి కాదు |
| తిరస్కరించలేని సైడ్ ఎఫెక్ట్లు | సాధ్యమే (చర్య ఇప్పటికే జరిగిపోయింది) | నివారించబడింది |
| ఆడిట్ స్పష్టత | ఆమోదం ఒక ప్రింట్ స్టేట్‌మెంట్ | ఆమోదం టైం స్టాంప్, చర్య, కారణంతో కూడిన JSON రికార్డు |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## షైలి 2: ప్రమాదం స్థాయి వర్గీకరణ

ప్రతి చర్యకూ మానవ అనుమతి అవసరం ఉండదు. ఒక పబ్లిక్ APIపై రీడ్-ఒన్లీ లుకప్ చేసే ప్రక్రియకు కస్టమర్‌కు ఇమెయిల్ పంపించడం కంటే భిన్నమైన ప్రాధాన్యతలు ఉంటాయి. రెండింటినీ ఒకటే విధంగా తీసుకోవడం ఆపరేటర్‌ ధ్యానం వృథా చేస్తుంది మరియు ఏజెంట్ స్లోగా చేస్తుంది.

సాధారణ 3-స్థాయి నమూనా:

| స్థాయి | ఉదాహరణలు | అనుమతి ఫ్లో |
|---|---|---|
| `low` (పఠనం మాత్రమే) | జ్ఞాన ఆదేశం శోధించడం, ఫ్లైట్ ఆప్షన్లను లెక్కించడం, పబ్లిక్ వెబ్ పేజీని తీసుకువెళ్ళడం | ఆటో-నిర్వహణ, ఆడిట్ కోసం లాగ్‌ చేయబడింది |
| `medium` (తక్కువ ఖర్చు మార్పు) | ఫలితాన్ని క్యాష్ చేయడం, కౌంటర్‌ను పెంచడం, రిమైండర్‌ను షెడ్యూల్ చేయడం | ఆటో-నిర్వహణ, కానీ రోజువారీ సమీక్షలో చేర్చబడింది |
| `high` (బాహ్య ముఖాముఖి లేదా తిరిగి మోసుకోలేని) | ఇమెయిల్ పంపడం, కార్డ్ చార్జ్ చేయడం, పబ్లిక్ ఛానెల్‌కు పోస్ట్ చేయడం | మానవ అనుమతిపై నిషేధించబడింది |

ఇది ఒక స్థాయి వర్గీకరణ మాత్రమే. ఉత్పత్తి వ్యవస్థలు ఎక్కువ సూక్ష్మ స్థాయిలను ఉపయోగిస్తాయి (ఉదాహరణకు, AWS IAM అనుమతి స్థాయిలు, పాత్ర ఆధారిత ప్రవేశ స్థాయిలు). క్రింద 3-స్థాయి వర్గీకరణ చదవడం మాత్రమే మరియు పక్క ఫలితాలు కలిపే చర్యలను మిక్స్ చేసే ఏజెంట్‌కోసం అత్యల్ప ఉపయోగపడే సంస్కరణ.

క్లాసిఫయర్ క్రింది విధంగా కీవర్డ్ హెయురిస్టిక్‌లు ఉపయోగిస్తుంది కాబట్టి డెమో నిర్దిష్టంగా మరియు తక్కువ ఖర్చుతో ఉంటుంది. ఉత్పత్తి వ్యవస్థలో దీన్ని ఒక నేర్చుకున్న క్లాసిఫయర్ లేదా విధాన ఇంజిన్‌తో మార్చవచ్చు.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## నమూనా 3: ఆడిట్ లాగ్ మరియు సవరణ లూప్

`print("Response approved.")` అనేది ఆడిట్ లాగ్ కాదు. విశ్వాసానికై, ప్రతి గేట్ నిర్ణయాన్ని మీరు తర్వాత పరిశీలించడానికి, పునరావృతంగా ఆడేందుకు లేదా ఘటన సమీక్షకు జతచేయడానికి ఉన్న ఒక నిర్మిత ఈవెంట్‌గా రికార్డ్ చేయాలి.

రెండు భాగాలు:

1. **బహిష్కరణ మాత్రమే JSONL.** ప్రతి నిర్ణయానికి ఒక్క లైన్తో, టైమ్స్టాంప్, చర్య, స్థాయి, నిర్ణయం, కారణం. grep చేయడం సులభం, తర్వాత నిజమైన లాగ్ స్టోర్‌కు పంపించడం సులభం.
2. **విమర్శలపై సవరణ లూప్.** గేట్ `deny` తిరిగి ఇచ్చినప్పుడు, ఏజెంట్ తిరిగి ఆ నిరాకరణ కారణంతో సుడిగాలి చేసుకుంటుంది, తద్వారా తదుపరి ప్రతిపాదన సమస్యను తప్పించుకునే అవకాశం కలుగుతుంది.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## అదనపు వనరులు

ఇతర పబ్లిక్ ప్రాజెక్ట్స్ ఈ HITL నమూనాల మామూలు మార్చివివిధ రూపాలతో అమలుపరచబడ్డాయి. మీ స్టాక్‌కి సరిపోయే దాన్ని కనుగొనడానికి పద్ధతులు సరిపోల్చండి:

- **LangChain** human-in-the-loop టూల్ రాపర్లు ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): మానవ ఇన్పుట్ కోసం ఎగ్జిక్యూషన్‌ను నిలిపే డ్రాప్-ఇన్ టూల్ రాపర్లు.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ దీన్ని పునఃసంరచించింది): బహుళ-ఏజెంట్ సంభాషణల్లో మానవుని ప్రతినిధానం చేయడానికి agent పాత్రను ఉపయోగిస్తుంది.
- **Semantic Kernel** ఫంక్షన్ ఫిల్టర్స్ ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): ప్రతి ఫంక్షన్ కాల్ చుట్టూ నడిచే మిడిల్‌వేర్, గేటింగ్ లాజిక్‌కు అనుకూలంగా ఉంటుంది.

ప్రతి ప్రాజెక్ట్ ఈ మూడు ఉప-నమూనాలను భిన్నంగా నిర్వహిస్తుంది: LangChain వాటిని టూల్స్‌గా రాప్స్ చేస్తుంది, AutoGen ఏజెంట్ పాత్రను ఉపయోగిస్తుంది, Semantic Kernel మిడిల్‌వేర్ ఫిల్టర్స్‌ను ఉపయోగిస్తుంది. మీ స్వంత ఏజెంట్ డిజైన్ ఎంచుకునే ముందు ఒకటి లేదా రెండు అమలు దశలను పూర్తిగా చదవండి.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
